In [2]:
from spark_utils import set_job_context

# Python, Spark and JVM version

In [12]:
import sys
import pyspark
import py4j

print(sys.executable)
print("PySpark:", pyspark.__version__)
print("Py4J:", py4j.__version__)

# Create Spark Session

In [13]:
import pyspark

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (
    SparkSession.builder
    .appName("DeltaLab")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:/workspace/spark-events")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Turning on the AQE (Adaptive Query Execution)

In [6]:

#Uncomment the code the turn it on
# spark.conf.set("spark.sql.adaptive.enabled", "true")

# Sample spark code 

In [7]:
from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Create a reasonably large distributed dataset
# --------------------------------------------------

df = (
    spark.range(0, 5_000_000, 1, numPartitions=8)
    .withColumn("group_id", (F.col("id") % 1000).cast("int"))
    .withColumn("value", (F.col("id") * 10) % 5000)
)

print("Initial partitions:", df.rdd.getNumPartitions())


# --------------------------------------------------
# 2. Force a shuffle using repartition
# --------------------------------------------------

repartitioned = df.repartition(16, "group_id")

print("After repartition:", repartitioned.rdd.getNumPartitions())


# --------------------------------------------------
# 3. Create a second dataset for a join
# --------------------------------------------------

lookup = (
    spark.range(0, 1000)
    .withColumnRenamed("id", "group_id")
    .withColumn(
        "category",
        F.concat(F.lit("category_"), F.col("group_id"))
    )
)


# --------------------------------------------------
# 4. Disable broadcast so Spark performs a shuffle join
# --------------------------------------------------

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

joined = repartitioned.join(
    lookup,
    on="group_id",
    how="inner"
)


# --------------------------------------------------
# 5. Aggregate
# --------------------------------------------------

aggregated = (
    joined
    .groupBy("category")
    .agg(
        F.count("*").alias("row_count"),
        F.sum("value").alias("total_value"),
        F.avg("value").alias("avg_value")
    )
)


# --------------------------------------------------
# 6. Sort
# --------------------------------------------------

result = aggregated.orderBy(
    F.desc("total_value")
)


# --------------------------------------------------
# 7. Write result as Delta
# --------------------------------------------------

output_path = "/workspace/data/delta/category_summary"

# --------------------------------------------------
# 8. Set job
# This helps to track this individual call in the spark UI,
# otherwise it will have functions name by default hard to distinguish between different runs
# --------------------------------------------------
job_id = set_job_context(
    spark,
    "Join, aggregate, sort, and write category summary as Delta"
)

result.write \
    .format("delta") \
    .mode("overwrite") \
    .save(output_path)

print(f"Delta table written to: {output_path}")

# Read the Delta tables 

In [14]:
#Change Job_id, to track what read opeartion did exactly
job_id = set_job_context(
    spark,
    "Read Delta table"
)

delta_df = (
    spark.read
    .format("delta")
    .load("/workspace/data/delta/category_summary")
)

delta_df.show(50, truncate=False)

# Close the spark session

In [10]:
# when finished with this Spark application
# spark.stop()